# Churn Prediction — Model Exploration

This notebook documents the thinking behind the model:
- What the data looks like
- Which features drive churn
- Why RandomForest was chosen
- How well it performs

The clean production version of this code lives in `ml/train.py`.

In [ ]:
import sys
sys.path.append('../..')  # so we can import from ml/train.py

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, ConfusionMatrixDisplay

# Import our data generator and pipeline builder from train.py
from ml.train import generate_churn_data, build_pipeline, ALL_FEATURES

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)

## 1. Generate and Explore the Dataset

In [ ]:
df = generate_churn_data(n_samples=5000)
print(f"Shape: {df.shape}")
print(f"Churn rate: {df['churn'].mean():.1%}")
df.head()

In [ ]:
df.describe()

## 2. Churn by Contract Type

This is the single strongest driver of churn — month-to-month customers leave much more.

In [ ]:
churn_by_contract = df.groupby('contract')['churn'].mean().sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Churn rate per contract
churn_by_contract.plot(kind='bar', ax=axes[0], color=['#e74c3c', '#f39c12', '#2ecc71'], edgecolor='black')
axes[0].set_title('Churn Rate by Contract Type', fontsize=13)
axes[0].set_ylabel('Churn Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30)
axes[0].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

# Distribution of contract types
df['contract'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue', edgecolor='black')
axes[1].set_title('Contract Type Distribution', fontsize=13)
axes[1].set_ylabel('Count')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)

plt.tight_layout()
plt.show()

## 3. Churn by Tenure

New customers churn much more. After ~36 months, churn drops significantly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Tenure distribution for churned vs retained
df[df['churn'] == 1]['tenure'].plot(kind='hist', bins=30, ax=axes[0],
                                    alpha=0.6, color='red', label='Churned')
df[df['churn'] == 0]['tenure'].plot(kind='hist', bins=30, ax=axes[0],
                                    alpha=0.6, color='green', label='Retained')
axes[0].set_title('Tenure Distribution: Churned vs Retained', fontsize=13)
axes[0].set_xlabel('Tenure (months)')
axes[0].legend()

# Churn rate by tenure bucket
df['tenure_bucket'] = pd.cut(df['tenure'], bins=[0, 6, 12, 24, 36, 72],
                              labels=['0-6m', '6-12m', '12-24m', '24-36m', '36m+'])
df.groupby('tenure_bucket')['churn'].mean().plot(kind='bar', ax=axes[1],
                                                  color='steelblue', edgecolor='black')
axes[1].set_title('Churn Rate by Tenure Bucket', fontsize=13)
axes[1].set_ylabel('Churn Rate')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30)
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))

plt.tight_layout()
plt.show()

## 4. Monthly Charges vs Churn

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
df[df['churn'] == 1]['monthly_charges'].plot(kind='hist', bins=30, ax=ax,
                                              alpha=0.6, color='red', label='Churned')
df[df['churn'] == 0]['monthly_charges'].plot(kind='hist', bins=30, ax=ax,
                                              alpha=0.6, color='green', label='Retained')
ax.set_title('Monthly Charges: Churned vs Retained', fontsize=13)
ax.set_xlabel('Monthly Charges ($)')
ax.legend()
plt.show()

## 5. Correlation Heatmap

In [ ]:
# Encode categoricals for correlation
df_encoded = df.copy()
df_encoded['contract_enc']  = df['contract'].map({'month-to-month': 2, 'one-year': 1, 'two-year': 0})
df_encoded['internet_enc']  = df['internet_service'].map({'Fiber': 2, 'DSL': 1, 'No': 0})
df_encoded['payment_enc']   = df['payment_method'].map({'electronic': 3, 'credit': 2, 'bank': 1, 'mail': 0})

cols = ['tenure', 'monthly_charges', 'total_charges', 'contract_enc',
        'internet_enc', 'payment_enc', 'senior_citizen', 'dependents', 'phone_service', 'churn']

plt.figure(figsize=(10, 8))
sns.heatmap(df_encoded[cols].corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('Feature Correlation Matrix', fontsize=13)
plt.tight_layout()
plt.show()

## 6. Train and Evaluate the Model

In [ ]:
X = df[ALL_FEATURES]
y = df['churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipeline = build_pipeline()
pipeline.fit(X_train, y_train)

y_pred  = pipeline.predict(X_test)
y_proba = pipeline.predict_proba(X_test)[:, 1]

print(f"ROC-AUC: {roc_auc_score(y_test, y_proba):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=['No Churn', 'Churn']))

## 7. Feature Importance

In [ ]:
rf_model     = pipeline.named_steps['classifier']
feature_names = ALL_FEATURES
importances   = rf_model.feature_importances_

fi_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
fi_df = fi_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
fi_df.plot(kind='barh', x='feature', y='importance', ax=ax,
           color='steelblue', edgecolor='black', legend=False)
ax.set_title('Feature Importances — RandomForest', fontsize=13)
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

## 8. Why RandomForest?

- Handles both numeric and categorical features naturally
- `class_weight='balanced'` handles the class imbalance (more non-churners than churners)
- Returns calibrated probabilities via `predict_proba()` — needed for the risk tiers (Low/Medium/High)
- Interpretable via feature importances
- No hyperparameter tuning needed for a production baseline

**What you'd do next in a real company:**
Compare against XGBoost/LightGBM, tune thresholds based on business cost of false negatives vs false positives, and add SHAP for per-prediction explainability.